# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities (record sets, fields, etc.) referenced explicitly by their `@id` for clarity and reproducibility.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).

In [ ]:
# Ensure `mlcroissant` is installed. Uncomment the next line if running in a fresh environment.
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata
meta = dataset.metadata

print(meta.name + ": " + meta.description)
print(f"Published: {meta.datePublished}\nVersion: {getattr(meta, 'version', None)}")

## 2. Data Overview

Examine available record sets, fields, and their `@id`s. This helps identify which portions of the dataset are available for extraction and analysis.

In [ ]:
# List all record sets by their `@id`, title, and available fields (by `@id`)

record_sets = list(dataset.record_sets())
if not record_sets:
    print("No explicit record sets declared under `recordSet` in dataset metadata. Attempting to infer from files").
    # Optionally, infer from the available DataFiles. mlcroissant will commonly provide a default record set if only one exists
    record_sets = dataset.record_sets(infer=True)

for record_set in record_sets:
    print(f"RecordSet @id: {record_set.id}")
    print(f"  Name: {record_set.name}")
    print(f"  Description: {record_set.description}")
    print(f"  Fields:")
    for field in record_set.fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, type: {field.data_type})")
    print()

## 3. Data Extraction

For demonstration purposes, we load the survey data from the main record set into a pandas DataFrame. Make sure to reference all entities using their `@id` only.

> **Note:** If only one record set is present (as is typical for tabular survey data), it can be loaded directly.

In [ ]:
# Get main record set by @id (usually visible above)

record_sets = list(dataset.record_sets())
if len(record_sets) == 1:
    main_record_set_id = record_sets[0].id
else:
    # For this dataset, the record set is likely the tabular survey records. Otherwise, select the appropriate one after overview.
    main_record_set_id = record_sets[0].id  # fallback to first
print(f"Using record set @id: {main_record_set_id}\n")

# List field @ids for the selected record set
field_ids = [f.id for f in record_sets[0].fields]
print("Field @ids:", field_ids)

# Load the records as a DataFrame
records = list(dataset.records(record_set=main_record_set_id))
df = pd.DataFrame(records)
# Preview column names (-- all @id for traceability)
print("Columns:", df.columns.tolist())
df.head()

## 4. Exploratory Data Analysis (EDA)

Let's conduct some basic data cleaning and EDA. We will use only `@id` values to reference columns and groupings, as required.

In [ ]:
# Identify a key numeric field for EDA. Here, we use the GAD-7 score if available.
# Find a field whose @id contains "gad7" or "GAD7" or "gad_7" (common for survey datasets)

import re
def find_field_id(pattern, cols):
    for col in cols:
        if re.search(pattern, col, re.IGNORECASE):
            return col
    return None

gad7_field_id = find_field_id("gad.?7", df.columns)
phq9_field_id = find_field_id("phq.?9", df.columns)
psq_field_id = find_field_id("psq", df.columns)

numeric_field_id = gad7_field_id or phq9_field_id or psq_field_id
if numeric_field_id is None:
    # fallback: pick the first float/int
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break

print(f"Numeric field @id for analysis: {numeric_field_id}")
print(df[[numeric_field_id]].describe())

# Remove outliers (if any) and filter for demonstration (e.g., GAD-7 score > 10 for potential moderate/severe anxiety)
threshold = 10
filtered_df = df[df[numeric_field_id] > threshold].copy()
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Attempt group by another key field, such as village or gender
village_field_id = find_field_id("village", df.columns)
gender_field_id = find_field_id("gender", df.columns)
group_field_id = village_field_id or gender_field_id

if group_field_id and group_field_id in df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization

Visualize the distribution of scores (e.g., GAD-7, PHQ-9, or PSQ) and their grouping by key attributes. Feel free to extend with more advanced charts (histograms, bar charts, boxplots, etc.).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the score distribution
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=15, color="skyblue")
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel('Count')
plt.show()

if group_field_id:
    plt.figure(figsize=(10,5))
    sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion

In this notebook, we loaded, explored, and visualized the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the Croissant schema and the `mlcroissant` library. We referenced all major dataset entities using their `@id`, enabling reproducible and standards-based data science workflows.

**Key findings:**
- The dataset contains rich demographic and mental health survey data, with key validated scale scores (such as GAD-7, PHQ-9, etc.).
- Preliminary EDA reveals variation in mental health scores across different subgroups (e.g., by village or gender).

This exploratory workflow is a foundation for further AI-ready modeling, statistical analysis, and community health research.